# Stockage — Data Lake et Base de données

Le stockage est la dernière étape du pipeline. Les données
transitent par deux outils complémentaires :

**MinIO — le Data Lake**
Reçoit les données BRUTES depuis le topic Redpanda `capteurs_agri`
(les données de Thierno, avant tout traitement Spark).
On les stocke en format Parquet — format compressé standard
du Big Data. MinIO utilise le même protocole qu'Amazon S3 :
en production réelle on remplacerait MinIO par S3 en changeant
deux lignes de configuration.

**PostgreSQL — la base structurée**
Reçoit les données TRAITÉES depuis le topic Redpanda `alertes_agri`
(la sortie de Spark de Fatou, déjà nettoyée et enrichie avec les alertes).
Ces données sont stockées en table SQL, directement requêtables.
C'est depuis PostgreSQL que le dashboard de l'agriculteur sera alimenté.

**Pourquoi les deux ?**
MinIO stocke tout en brut pour pouvoir retraiter plus tard si besoin.
PostgreSQL stocke le résultat final pour pouvoir l'interroger
immédiatement en SQL. Ce sont deux besoins différents,
deux outils différents.

## Étape 1 — Stockage des données brutes dans MinIO

In [ ]:
exec(open("../storage/minio_upload.py").read())
# Output attendu :
# 10 messages récupérés depuis capteurs_agri
# Upload MinIO réussi.

## Étape 2 — Stockage des données traitées dans PostgreSQL

In [ ]:
exec(open("../storage/postgres_insert.py").read())
# Output attendu :
# 10 messages récupérés depuis alertes_agri
# Insertion PostgreSQL réussie.

## Vérification — Requête SQL sur les données stockées

On vérifie que les données sont bien dans PostgreSQL
en faisant une requête directement depuis le notebook.
C'est exactement ce que ferait un dashboard en production.

In [ ]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="agridb", user="postgres", password="postgres"
)

df_result = pd.read_sql("SELECT * FROM capteurs", conn)
print(df_result)
conn.close()

# Output attendu :
#   capteur_id  temperature  humidite   ph    timestamp   alerte
# 0       C001         34.2      18.5  6.3   1234567890       OK
# 1       C002         29.1      45.0  7.1   1234567891       OK


In [ ]:
conn = psycopg2.connect(
    host="localhost", port=5432,
    database="agridb", user="postgres", password="postgres"
)

# Exemple de requête que l'agriculteur ferait via le dashboard
df_alertes = pd.read_sql("""
    SELECT * FROM capteurs
    WHERE alerte != 'OK'
""", conn)

print("Alertes actives :")
print(df_alertes)
conn.close()

## Idée complémentaire — Partitionnement par date et région

En production avec des milliers de capteurs, on partitionnerait
les fichiers MinIO par date et par région :

    agri-data/
    ├── region=Dakar/
    │   ├── date=2025-05-01/donnees_brutes.parquet
    │   └── date=2025-05-02/donnees_brutes.parquet
    └── region=Thiès/
        └── date=2025-05-01/donnees_brutes.parquet

Avantage : Spark ne lit que la partition utile au lieu de
tout le Data Lake — requêtes 10x plus rapides sur gros volumes.